# Module 7: Handling Errors

**Utrains Python Fundamentals** &middot; lab notebook

*Catch problems instead of crashing, retry what's worth retrying, and raise your own errors.*

## By the end of this notebook you can

- Explain what happens when Python raises an error
- Catch a specific error with try and except
- Use multiple except blocks, plus else and finally
- Retry a flaky call instead of giving up on the first failure
- Raise your own error when your code detects a problem

## How to work through it

It follows the Module 7 slide deck, slide by slide.

The headings below are the slide numbers from the deck. The explanation for
each one is on the slide and in the [README](../README.md); this notebook is
where you run the code.

Run every cell in order with **Shift + Enter**.

Two cells are marked **Your turn**. They contain `____` where a piece of the
syntax is missing, so they fail if you run them as they are. That is
deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**: a short task with no code written for you.

**Assumed knowledge.** Modules 1 to 6. The retry slide uses `def` to build a function, exactly as the deck does. Functions get their full treatment in Module 8; here you only need to read one, not design one.

## Slide 2 &middot; When Something Goes Wrong

In [ ]:
# The crashing version, kept safe inside a demonstration.
text = "abc"

try:
    number = int(text)
except ValueError as e:
    print("Python raised:", type(e).__name__)
    print("with the message:", e)

print("and the program carried on")

## Slide 3 &middot; try and except

In [ ]:
text = "abc"

try:
    number = int(text)
except ValueError:
    print("That was not a number.")

In [ ]:
# The same shape, applied to a truncated API response.
import json

broken_response = '{"id": "msg_01", "content": '

try:
    data = json.loads(broken_response)
    print("parsed fine:", data["content"])
except json.JSONDecodeError:
    print("Failed to parse API response.")

> **Heads up.** `import` appears here because JSON parsing needs it. Module 10 explains `import` in full. The short version: it brings in code someone else already wrote, used with a dot, like `json.loads()`.

## Slide 4 &middot; Multiple except, else, and finally

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero.")
except ValueError:
    print("Not a valid value.")
else:
    print("Success:", result)      # only if no exception
finally:
    print("Done trying.")          # always runs

Change `10 / 0` to `10 / 2` and run the cell again. The `else` branch fires, and `finally` still runs. That is the whole point of the pair.

---

### Your turn 1

An incident ticket must always end up marked closed, even when the update step fails. Pick the two keywords that catch the failure and guarantee the closing step.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
ticket = {"id": "INC-4412", "status": "open"}

# TODO: catch the failure, then guarantee the closing step always runs.
try:
    raise ConnectionError("ticketing system unreachable")
____ ConnectionError as e:
    print("could not update:", e)
____:
    ticket["status"] = "closed"

print(ticket)

## Slide 5 &middot; Retrying Instead of Giving Up

In [ ]:
import time


def call_model(prompt):
    if prompt == "":
        raise TimeoutError("model did not respond in time")
    return f"response to: {prompt}"


def call_with_retry(prompt, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return call_model(prompt)
        except TimeoutError:
            print(f"attempt {attempt} timed out, retrying...")
            time.sleep(0.2)
    raise RuntimeError("model call failed after all retries")


print(call_with_retry("Summarize this document."))

In [ ]:
try:
    call_with_retry("")
except RuntimeError as e:
    print("gave up:", e)

## Slide 6 &middot; Raising Your Own Errors

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError("age cannot be negative")
    return age


try:
    set_age(-5)
except ValueError as e:
    print("Invalid input:", e)

---

### Your turn 2

Guard a model temperature setting. Anything outside 0.0 to 2.0 should signal a problem with a clear message, and the caller should catch it and read that message.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
def set_temperature(value):
    if value < 0.0 or value > 2.0:
        # TODO: signal the problem yourself, with the right error type.
        ____ ____(f"temperature {value} is outside 0.0 to 2.0")
    return value


print("valid:", set_temperature(0.7))

# TODO: catch it, and capture the message so it can be printed.
try:
    set_temperature(3.5)
except ValueError ____ e:
    print("rejected:", e)

---

# More use cases

The same ideas, applied to situations you will meet in real work. Run each one, then change a value and run it again.

## Use case 1 &middot; AI &middot; A response that is not the shape you expected

In [ ]:
def read_reply(response):
    try:
        return response["choices"][0]["message"]["content"]
    except KeyError as e:
        return f"missing key in the response: {e}"
    except IndexError:
        return "the response came back with no choices"
    except TypeError:
        return "that was not shaped like a response at all"


good = {"choices": [{"message": {"content": "All targets healthy."}}]}
missing = {"choices": [{"message": {}}]}
empty = {"choices": []}

print(read_reply(good))
print(read_reply(missing))
print(read_reply(empty))
print(read_reply("just a string"))

## Use case 2 &middot; AI &middot; Retry the model, then give up honestly

In [ ]:
import time

calls = {"n": 0}


def call_model(prompt):
    calls["n"] += 1
    if calls["n"] < 3:
        raise TimeoutError("model did not respond in time")
    return f"answer to: {prompt}"


def ask(prompt, attempts=4):
    for attempt in range(1, attempts + 1):
        try:
            return call_model(prompt)
        except TimeoutError as e:
            print(f"  attempt {attempt}: {e}")
            time.sleep(0.1)
    raise RuntimeError(f"gave up after {attempts} attempts")


print(ask("summarise the incident"))
print("calls made:", calls["n"])

## Use case 3 &middot; Parse a value that might be junk

In [ ]:
def read_port(text, default=8080):
    try:
        return int(text)
    except ValueError:
        print(f"  could not read {text!r} as a port, using {default}")
        return default


print("8080       ->", read_port("8080"))
print("not-a-port ->", read_port("not-a-port"))
print("empty      ->", read_port(""))

## Use case 4 &middot; Always release the lock

In [ ]:
lock_held = False


def deploy_with_lock(should_fail):
    global lock_held
    lock_held = True
    print("  lock acquired")
    try:
        if should_fail:
            raise RuntimeError("deployment failed halfway")
        print("  deployment finished")
    except RuntimeError as e:
        print("  caught:", e)
    finally:
        lock_held = False
        print("  lock released")


deploy_with_lock(should_fail=False)
print("lock still held?", lock_held)

deploy_with_lock(should_fail=True)
print("lock still held?", lock_held)

---

## Lab: A deployment run that refuses to crash

Write a function `deploy(stage)` that raises a `RuntimeError` when the stage
name is `"migrate"`, and otherwise returns a success message.

Loop over the stages `build`, `test`, `migrate`, `release` and call `deploy()`
for each one. A failing stage must not stop the others.

Record every outcome in a results dictionary. Use `finally` so a line is always
printed for each stage, whether it succeeded or not. At the end, print how many
stages succeeded and how many failed.

For extra credit, wrap the `migrate` stage in the retry pattern from slide 5,
giving it three attempts before recording it as failed.

**Done when:**

- [ ] deploy() raises for one specific stage
- [ ] One failing stage does not stop the loop
- [ ] finally guarantees a line per stage
- [ ] A results dictionary records every outcome
- [ ] A summary counts successes and failures

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
stages = ["build", "test", "migrate", "release"]

# Your lab answer goes here.

---

## Practice exercises

The four exercises from the module's practice slide are in the
[README](../README.md#practice-exercises) and repeated on the slide. There are
4 of them. Do them in a scratch cell here or in a `.py` file.

## Module complete

You can now catch errors, retry what's worth retrying, and raise your own.

*Utrains &middot; support@utrains.org &middot; https://utrains.org*